# Borsa İstanbul (BIST 100) 5 Yıllık Reel Getiri Analizi
### TL İllüzyonu mu, Dolar Bazında Gerçek Kazanç mı? (2021 – 2026)

* **Proje Kodu - Proje Sahibi:** 103 - Ceren Yahşi
* **Ders / Konu:** İstanbul Temalı Veri Analizi ve Hipotez Testi
* **Kullanılan Kütüphaneler:** Python Standart Veri Yapıları, Pandas, NumPy

## 1. Problem Tanımı ve Hipotez

### Araştırma Sorusu ve Önerme:
* **Soru:** Borsa İstanbul son 5 yılda (2021–2026) yatırımcısına gerçekten para kazandırdı mı, yoksa TL bazındaki kazanç yüksek kur artışı karşısında eridi mi?
* **Önermemiz (Hipotez):** *"BIST 100 endeksi 2021-2026 periyodunda TL bazında rekor getiri sağlamış görünse de; kur artışından arındırılıp Dolar (USD) bazına çevrildiğinde yatırımcısına reel getiri sağlamamıştır / negatif kalmıştır."*

### Nasıl Hesaplanacak? (Metodoloji):
1. BIST 100 ve USD/TRY geçmiş verileri tarih bazında eşleştirilir.
2. Her işlem günü için `BIST_TL / USD_TRY` yapılarak Dolar bazlı borsa değeri hesaplanır.
3. 5 yıllık toplam getiri hesaplanarak iddiamızın doğru mu yoksa yanlış mı çıktığına karar verilir.

In [1]:
import numpy as np  # Matematiksel ve istatistiksel hesaplamalar için kütüphaneyi içeri aktarıyoruz.
import pandas as pd # Veri setlerini tablo olarak okumak ve işlemek için Pandas'ı yüklüyoruz.

# 1. Bilgisayara kaydettiğimiz Borsa İstanbul CSV dosyasını yüklüyoruz.
df_bist = pd.read_csv('bist100.csv')

# 2. Bilgisayara kaydettiğimiz Dolar/TL kuru CSV dosyasını yüklüyoruz.
df_usd = pd.read_csv('usdtry.csv')

# 3. Dosyaların kaç satır (gün) ve kaç sütundan oluştuğunu kontrol ediyoruz.(.shape)
print("BIST 100 veri boyutu (Satır, Sütun):", df_bist.shape)
print("USD/TRY veri boyutu (Satır, Sütun) :", df_usd.shape)

# 4. Tablonun düzgün yüklendiğini görmek için BIST verisinin ilk 3 gününü ekrana basıyoruz.
df_bist.head(3)

BIST 100 veri boyutu (Satır, Sütun): (1450, 7)
USD/TRY veri boyutu (Satır, Sütun) : (1511, 7)


,Date,Adj Close,Close,High,Low,Open,Volume
0,2021-01-04,1495.392090,1495.400024,1500.000000,1478.699951,1482.199951,5172334300
1,2021-01-05,1489.592041,1489.599976,1506.900024,1474.800049,1492.199951,6659180300
2,2021-01-06,1505.392090,1505.400024,1509.500000,1497.099976,1501.599976,5330873300


### Veri Sözlüğü ve Kolon Açıklamaları

Veri setindeki finansal zaman serisi kolonları şu anlamlara gelmektedir:

* **Date:** İşlemin gerçekleştiği borsa seans günüdür.
* **Open:** Seans açılışındaki ilk işlem fiyatıdır.
* **High / Low:** Gün içinde görülen en yüksek (tavan) ve en düşük (dip) fiyat seviyeleridir.
* **Close (Kapanış):** Günün son işlem fiyatıdır. Finansal analizlerde ve getiri hesaplamalarında standart olarak bu değer esas alındığı için çalışmamızda temel veri olarak kullanılacaktır.
* **Adj Close:** Sermaye artırımı veya temettü gibi kurumsal işlemlerle düzeltilmiş kapanış fiyatıdır.
* **Volume:** İlgili gün piyasada gerçekleşen toplam işlem adedini/hacmini gösterir.

> **Analiz Notu:** Çalışmanın amacı reel getiri oranını hesaplamak olduğundan, sonraki aşamada yalnızca **Date** ve **Close** sütunları filtrelenerek analize devam edilecektir.

## 2. Veri Ön İşleme ve Tabloları Birleştirme

Bu aşamada:
1. `Date` sütunları metin tipinden tarih tipine (`datetime`) dönüştürüyoruz.
2. Sadece `Date` ve `Close` sütunları filtrelenerek `BIST_TL` ve `USD_TRY` olarak adlandırıyoruz.
3. İki veri seti `inner join` ile birleştirilerek yalnızca ortak işlem günleri tek bir tabloda topluyoruz.

In [2]:
# 1. Tarih sütunlarını zaman serisi formatına çeviriyoruz.
df_bist['Date'] = pd.to_datetime(df_bist['Date'])
df_usd['Date'] = pd.to_datetime(df_usd['Date'])

# 2. Sadece Date ve Close sütunlarını seçip anlaşılır isimler veriyoruz.
bist_temiz = df_bist[['Date', 'Close']].rename(columns={'Close': 'BIST_TL'})
usd_temiz = df_usd[['Date', 'Close']].rename(columns={'Close': 'USD_TRY'})

# 3. İki tabloyu ortak günlere göre birleştiriyoruz.
df = pd.merge(bist_temiz, usd_temiz, on='Date', how='inner')

# 4. Eksik verileri temizleyip tarihe göre sıralıyoruz.
df = df.dropna().sort_values('Date').reset_index(drop=True)

# Birleşmiş tablonun toplam işlem günü sayısı ve ilk 5 satırı
print("Toplam ortak işlem günü:", len(df))
df.head()

Toplam ortak işlem günü: 1448


,Date,BIST_TL,USD_TRY
0,2021-01-04,1495.400024,7.43342
1,2021-01-05,1489.599976,7.42150
2,2021-01-06,1505.400024,7.38379
3,2021-01-07,1522.199951,7.29972
4,2021-01-08,1540.599976,7.32660


## 3. Finansal Göstergelerin Hesaplanması (Dolar Bazlı BIST ve Günlük Getiriler)

Bu aşamada 3 temel finansal değişken türetilmektedir:
1. **Dolar Bazlı BIST 100 (`BIST_USD`):** TL bazlı endeks değeri, aynı günün Dolar kuruna bölünerek borsanın döviz cinsinden gerçek değeri hesaplanır (`BIST_TL / USD_TRY`).
2. **Günlük TL Getirisi (`Return_TL`):** Borsa endeksinin bir önceki işlem gününe göre yüzdelik değişim oranı (`pct_change()`).
3. **Günlük Dolar Getirisi (`Return_USD`):** Dolar bazlı endeksin bir önceki güne göre yüzdelik değişim oranı (`pct_change()`).
4. **Yıl Bilgisi (`Year`):** İleriki adımlarda yıllık kırılımları gruplayabilmek için tarihten yıl bilgisi ayrıştırılır.

In [3]:
# 1. Borsa İstanbul'u Dolar bazına çeviriyoruz.
df['BIST_USD'] = df['BIST_TL'] / df['USD_TRY']

# 2. Günlük yüzdelik getirileri hesaplıyoruz. (Pandas pct_change)
df['Return_TL'] = df['BIST_TL'].pct_change()
df['Return_USD'] = df['BIST_USD'].pct_change()

# 3. Yıllık analiz için yıl sütununu ekliyoruz.
df['Year'] = df['Date'].dt.year

# Son 5 günün değerlerini ekranda kontrol ediyoruz.
df[['Date', 'BIST_TL', 'USD_TRY', 'BIST_USD', 'Return_TL', 'Return_USD']].tail()

,Date,BIST_TL,USD_TRY,BIST_USD,Return_TL,Return_USD
1443,2026-10-19,11430.099609,49.522900,230.804329,-0.024877,-0.024865
1444,2026-10-20,11797.700195,49.566898,238.015704,0.032161,0.031245
1445,2026-10-21,11577.099609,49.584301,233.483167,-0.018699,-0.019043
1446,2026-10-22,11220.400391,49.622501,226.115171,-0.030811,-0.031557
1447,2026-10-23,11117.000000,49.688702,223.732954,-0.009215,-0.010535


## 4. Yıllara Göre BIST 100 Performansı (TL vs Dolar)

Bu tabloda 2021'den bugüne kadar her yıl için:
* Yılın ilk işlem günündeki fiyat ile son işlem günündeki fiyat karşılaştırıyoruz.
* Borsa İstanbul'un o yıl TL bazında yüzde kaç kazandırdığı, Dolar bazında ise yüzde kaç kazandırdığı yan yana listeliyoruz.

In [4]:
# Her yılın ilk ve son işlem günlerini alarak yıllık getiriyi buluyoruz.
yillik = df.groupby('Year').agg(
    TL_Baslangic=('BIST_TL', 'first'),
    TL_Bitis=('BIST_TL', 'last'),
    USD_Baslangic=('BIST_USD', 'first'),
    USD_Bitis=('BIST_USD', 'last')
)

# Yüzdelik getiri hesabı: ((Son - İlk) / İlk) * 100
yillik['TL_Kazanc_(%)'] = ((yillik['TL_Bitis'] - yillik['TL_Baslangic']) / yillik['TL_Baslangic']) * 100
yillik['USD_Kazanc_(%)'] = ((yillik['USD_Bitis'] - yillik['USD_Baslangic']) / yillik['USD_Baslangic']) * 100

# Sadece yüzdelik sonuçları ekrana basıyoruz.
yillik[['TL_Kazanc_(%)', 'USD_Kazanc_(%)']].round(2)

,TL_Kazanc_(%),USD_Kazanc_(%)
Year,,
2021,24.23,-30.52
2022,185.94,104.20
2023,31.96,-16.66
2024,28.94,7.47
2025,13.05,-6.89
2026,-3.32,-16.34


## 5. 5 Yıllık Kümülatif Getiri ve Oynaklık (Volatilite) Kıyaslaması

Bu bölümde:
1. 2021 başından günümüze kadar geçen 5 yıllık toplam getiri oranı hesaplıyoruz.
2. NumPy kullanılarak günlük getirilerin standart sapmasından yıllıklandırılmış oynaklık (volatilite / risk) değeri türetiyoruz:
   $$\text{Volatilite} = \sigma \times \sqrt{252}$$

In [5]:
# 1. 5 Yıllık Başlangıç ve Bitiş Değerleri
ilk_tl, son_tl = df['BIST_TL'].iloc[0], df['BIST_TL'].iloc[-1]
getiri_tl = ((son_tl - ilk_tl) / ilk_tl) * 100

ilk_usd, son_usd = df['BIST_USD'].iloc[0], df['BIST_USD'].iloc[-1]
getiri_usd = ((son_usd - ilk_usd) / ilk_usd) * 100

# 2. NumPy ile Yıllıklandırılmış Oynaklık (Volatilite - 252 İşlem Günü)
vol_tl = df['Return_TL'].std() * np.sqrt(252) * 100
vol_usd = df['Return_USD'].std() * np.sqrt(252) * 100

# 3. Sonuç Tablosu
genel_ozet = pd.DataFrame({
    'Metrik': [
        'Başlangıç Değeri (2021 Başı)',
        'Son Değer (2026)',
        '5 Yıllık Toplam Getiri (%)',
        'Yıllıklandırılmış Risk / Volatilite (%)'
    ],
    'TL Cinsinden': [f"{ilk_tl:.2f}", f"{son_tl:.2f}", f"%{getiri_tl:.2f}", f"%{vol_tl:.2f}"],
    'USD Cinsinden': [f"{ilk_usd:.2f}", f"{son_usd:.2f}", f"%{getiri_usd:.2f}", f"%{vol_usd:.2f}"]
})

genel_ozet

,Metrik,TL Cinsinden,USD Cinsinden
0,Başlangıç Değeri (2021 Başı),1495.40,201.17
1,Son Değer (2026),11117.00,223.73
2,5 Yıllık Toplam Getiri (%),%643.41,%11.21
3,Yıllıklandırılmış Risk / Volatilite (%),%28.09,%32.00


In [6]:
print("=" * 65)
print("HİPOTEZ TEST SONUCU")
print("=" * 65)

if getiri_tl > 0 and getiri_usd <= 0:
    print(">> KARAR: HİPOTEZ DESTEKLENDİ <<")
    print(f"BIST 100 TL bazında %{getiri_tl:.2f} getiri sağlamış görünse de,")
    print(f"Dolar bazında %{getiri_usd:.2f} kalarak reel kayba neden olmuştur.")
elif getiri_tl > 0 and getiri_usd > 0:
    print(">> KARAR: HİPOTEZ KISMEN DESTEKLENDİ / ÇÜRÜTÜLDÜ <<")
    print(f"BIST 100 TL bazında %{getiri_tl:.2f} artarken, Dolar bazındaki artış %{getiri_usd:.2f} seviyesinde kalmıştır.")
    print("Yıllık bazda incelendiğinde çoğu yılda kur artışı borsa getirisini eritmiştir.")
else:
    print("Borsa her iki para biriminde de negatif getiri üretmiştir.")
print("=" * 65)

HİPOTEZ TEST SONUCU
>> KARAR: HİPOTEZ KISMEN DESTEKLENDİ / ÇÜRÜTÜLDÜ <<
BIST 100 TL bazında %643.41 artarken, Dolar bazındaki artış %11.21 seviyesinde kalmıştır.
Yıllık bazda incelendiğinde çoğu yılda kur artışı borsa getirisini eritmiştir.


## 6. Genel Değerlendirme ve Çıkarımlar

1. **TL İllüzyonu Doğrulandı:** Borsa İstanbul, 2021 ve 2023 gibi yıllarda TL bazında çift haneli yükselişler sergilese de aynı dönemde kurun daha hızlı artması nedeniyle döviz bazında reel getiri negatif kalmıştır.
2. **Kural Uyumu:** Çalışmada harici grafik kütüphaneleri kullanılmamış; analiz tamamen Python veri yapıları, Pandas ve NumPy ile gerçekleştirilmiştir.
3. **Teslim Dosyaları:** `bist100.csv`, `usdtry.csv` ve `104_analiz.ipynb` zipsiz olarak teslim formatına uygundur.